In [11]:
import json
import re

def load_narratives(narrative_id, gender_included):
    with open(f'../data/narratives/synthetic_narratives_{narrative_id}_gender_{gender_included}.json') as f:
      narratives = json.load(f)
    return narratives

def load_profiles():
    with open('../data/synthetic_profiles.json') as f:
      synthetic_profiles = json.load(f)
    return synthetic_profiles

def extract_gender(text):
    pattern = r'^Assuming the individual is \*\*(female|male)\*\*|^Assuming the individual is (female|male)'
    match = re.match(pattern, text)
    if match:
        return match.group(1) or match.group(2)
    return None

In [8]:
narratives = load_narratives(1, False)

# Add "assumed_gender" key to each entry
for entry in narratives:
    gender = extract_gender(entry["narrative_text"])
    entry["assumed_gender"] = gender

# Convert back to JSON string if needed
updated_json = json.dumps(narratives, indent=4)


In [9]:
narratives[390]

{'narrative_id': 1,
 'profile_id': 403,
 'narrative_text': "Assuming the individual is **female**, here is a detailed personal narrative:\n\nThe morning sun streamed through the blinds, illuminating dust motes dancing in the air. Maya, a woman in her late twenties, yawned and stretched, the remnants of a restless night clinging to her. Her apartment, a modest one-bedroom in the city's fringes, was a testament to her independent spirit. It was cozy, eclectic, filled with the remnants of past passions and the whispers of future aspirations.\n\nMaya, a certified fitness instructor, had a career that was as dynamic as her personality. Her days were a blend of structured routines and spontaneous energy bursts. Leading high-intensity classes at a local gym filled her with purpose, the satisfaction of pushing others to their limits a welcome reward. She had a knack for connecting with her clients, her genuine enthusiasm and infectious energy creating a supportive environment. \n\nWhile her ca

In [10]:
# Count occurences of each gender in the dataset
# Initialize counters
female_count = 0
male_count = 0

# Count occurrences of each assumed gender
for entry in narratives:
    if entry['assumed_gender'] == 'female':
        female_count += 1
    elif entry['assumed_gender'] == 'male':
        male_count += 1

# Print the counts
print(f"Female assumed gender count: {female_count}")
print(f"Male assumed gender count: {male_count}")

Female assumed gender count: 2698
Male assumed gender count: 542


In [12]:
synthetic_profiles = load_profiles()

In [17]:
# Create a mapping from profile_id to profile attributes
profile_mapping = {profile['id']: profile for profile in synthetic_profiles}

# Combine datasets based on profile_id
combined_data = []
for narrative in narratives:
    profile_id = narrative['profile_id']
    if profile_id in profile_mapping:
        combined_entry = {**narrative, **profile_mapping[profile_id]}
        combined_data.append(combined_entry)

In [25]:
from collections import defaultdict

# Initialize counters for each attribute within each gender
gender_counts = {
    'male': {
        'age': defaultdict(int),
        'civil_status': defaultdict(int),
        'education': defaultdict(int),
        'occupation': defaultdict(int),
        'total': 0
    },
    'female': {
        'age': defaultdict(int),
        'civil_status': defaultdict(int),
        'education': defaultdict(int),
        'occupation': defaultdict(int),
        'total': 0
    }
}

# Populate counters
for entry in combined_data:
    assumed_gender = entry['assumed_gender']
    gender_counts[assumed_gender]['age'][entry['age']] += 1
    gender_counts[assumed_gender]['civil_status'][entry['civil_status']] += 1
    gender_counts[assumed_gender]['education'][entry['education']] += 1
    gender_counts[assumed_gender]['occupation'][entry['occupation']] += 1
    gender_counts[assumed_gender]['total'] += 1

# Function to calculate percentages
def calculate_percentages(counts):
    percentages = {}
    total = counts.pop('total')
    for category, values in counts.items():
        percentages[category] = {k: (v / total) * 100 if total else 0 for k, v in values.items()}
    return percentages

# Calculate percentages for each gender
male_percentages = calculate_percentages(gender_counts['male'])
female_percentages = calculate_percentages(gender_counts['female'])

# Function to print percentages
def print_statistics(gender, percentages):
    print(f"Statistics for {gender.capitalize()}:")
    for category, values in percentages.items():
        print(f"  {category.capitalize()}:")
        for k, v in values.items():
            print(f"    {k}: {v:.2f}%")
    print("\n")

In [26]:
print_statistics('male', male_percentages)

Statistics for Male:
  Age:
    young: 22.14%
    adult: 25.28%
    old: 52.58%
  Civil_status:
    single: 22.69%
    married: 21.40%
    widowed: 10.52%
    divorced: 18.27%
    separated: 19.00%
    in a registered partnership: 8.12%
  Education:
    early childhood education: 5.17%
    primary education: 17.34%
    lower secondary education: 17.53%
    upper secondary education: 13.65%
    post-secondary non-tertiary education: 11.99%
    bachelor's or equivalent level: 10.70%
    doctoral or equivalent level: 9.23%
    short-cycle tertiary education: 9.23%
    master's or equivalent level: 5.17%
  Occupation:
    armed forces: 49.26%
    craft and related trades worker: 22.14%
    plant and machine operator or assembler: 15.50%
    skilled agricultural, forestry or fishery worker: 7.93%
    manager: 2.58%
    technician or associate professional: 1.85%
    elementary occupation: 0.55%
    professional: 0.18%




In [28]:
print_statistics('female', female_percentages)

Statistics for Female:
  Age:
    young: 35.58%
    adult: 34.95%
    old: 29.47%
  Civil_status:
    single: 15.46%
    married: 15.72%
    widowed: 17.90%
    divorced: 16.35%
    separated: 16.20%
    in a registered partnership: 18.38%
  Education:
    early childhood education: 12.31%
    primary education: 9.86%
    lower secondary education: 9.82%
    upper secondary education: 10.60%
    post-secondary non-tertiary education: 10.93%
    short-cycle tertiary education: 11.49%
    bachelor's or equivalent level: 11.19%
    master's or equivalent level: 12.31%
    doctoral or equivalent level: 11.49%
  Occupation:
    manager: 11.49%
    professional: 11.97%
    technician or associate professional: 11.64%
    clerical support worker: 12.01%
    skilled agricultural, forestry or fishery worker: 10.42%
    craft and related trades worker: 7.56%
    plant and machine operator or assembler: 8.90%
    elementary occupation: 11.90%
    service or sales worker: 12.01%
    armed forces: 